In [1]:
import warnings

# Suppress Pydantic V2 warnings BEFORE importing CrewAI
warnings.filterwarnings('ignore', category=DeprecationWarning, module='pydantic.*')
warnings.filterwarnings('ignore', category=UserWarning, module='pydantic.*')

from collections import Counter
from pprint import pprint

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

import os

import litellm
from dotenv import load_dotenv

from crewai import Agent, Task, Crew, Process, LLM

# Configure LiteLLM to handle Anthropic API message format requirements
litellm.modify_params = True

# Load environment variables (for API keys)
load_dotenv()

import watermark

%load_ext watermark
%matplotlib inline

We start by printing out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.10.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.2.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 2b98c2c19a22d9b5e74349315b8e0b33a6a32a32

crewai    : 0.86.0
numpy     : 1.26.4
matplotlib: 3.8.4
watermark : 2.5.0
pandas    : 2.2.3
litellm   : 1.60.0



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')

# 1. Foundations of Autonomous Agents

In this notebook, we'll explore the fundamental building blocks of CrewAI agents:
- **Roles**: What the agent is (e.g., "Research Analyst", "Content Writer")
- **Goals**: What the agent wants to accomplish
- **Backstories**: Context that shapes the agent's behavior and expertise

Understanding these core concepts is essential before building complex multi-agent systems.

Initialize the LLM - using Anthropic's Claude Sonnet 4.5. CrewAI uses LiteLLM internally, so we need the provider prefix

In [4]:
llm = LLM(
    model="anthropic/claude-sonnet-4-5-20250929",
    temperature=0.7
)

## Core Concept 1: Roles

A **role** defines what the agent is. It's like assigning a job title to your agent. The role should be:
- **Specific**: Clear and focused (e.g., "Senior Python Developer" not just "Developer")
- **Professional**: Reflects expertise level and domain
- **Relevant**: Matches the task you want the agent to perform

In [5]:
# Example: Create a simple research agent with a clear role
researcher = Agent(
    role="Senior Research Analyst",
    goal="Discover and analyze information about AI trends",
    backstory="You are an experienced research analyst with 10 years of experience "
              "in technology trends. You excel at finding accurate information and "
              "synthesizing it into actionable insights.",
    llm=llm,
    verbose=True
)

print(f"Agent Role: {researcher.role}")
print(f"Agent Goal: {researcher.goal}")

Agent Role: Senior Research Analyst
Agent Goal: Discover and analyze information about AI trends


## Core Concept 2: Goals

A **goal** defines what the agent wants to accomplish. Good goals are:
- **Action-oriented**: Start with verbs like "analyze", "create", "discover"
- **Measurable**: Clear success criteria
- **Aligned with role**: The goal should match the agent's expertise

In [6]:
# Example: Different agents with different goals
writer = Agent(
    role="Content Writer",
    goal="Create engaging and informative blog posts about technology",
    backstory="You are a skilled technical writer with a talent for making "
              "complex topics accessible to general audiences.",
    llm=llm,
    verbose=True
)

editor = Agent(
    role="Senior Editor",
    goal="Review and polish content for clarity, accuracy, and engagement",
    backstory="You are a meticulous editor with 15 years of experience in "
              "technical publishing. You have an eye for detail and ensure "
              "all content meets the highest standards.",
    llm=llm,
    verbose=True
)

print("Agent Goals:")
print(f"  Researcher: {researcher.goal}")
print(f"  Writer: {writer.goal}")
print(f"  Editor: {editor.goal}")

Agent Goals:
  Researcher: Discover and analyze information about AI trends
  Writer: Create engaging and informative blog posts about technology
  Editor: Review and polish content for clarity, accuracy, and engagement


## Core Concept 3: Backstories

A **backstory** provides context and personality to your agent. It:
- **Establishes expertise**: Credentials, experience, specializations
- **Sets behavior patterns**: How the agent approaches problems
- **Creates persona**: Makes the agent feel more realistic and consistent

The LLM uses the backstory to guide its responses and maintain character consistency.

In [7]:
# Example: Agents with detailed backstories
financial_analyst = Agent(
    role="Senior Financial Analyst",
    goal="Analyze market trends and provide investment insights",
    backstory="""
        You worked at Goldman Sachs for 12 years, specializing in tech sector analysis.
        You have a CFA designation and an MBA from Wharton. Your analysis has helped
        identify major market opportunities in AI, cloud computing, and fintech.
        
        You are known for:
        - Data-driven decision making
        - Conservative risk assessment
        - Clear, actionable recommendations
        - Citing specific sources and metrics
    """,
    llm=llm,
    verbose=True
)

print("Financial Analyst Profile:")
print(f"  Role: {financial_analyst.role}")
print(f"  Goal: {financial_analyst.goal}")
print(f"  Backstory (first 100 chars): {financial_analyst.backstory[:100]}...")

Financial Analyst Profile:
  Role: Senior Financial Analyst
  Goal: Analyze market trends and provide investment insights
  Backstory (first 100 chars): 
        You worked at Goldman Sachs for 12 years, specializing in tech sector analysis.
        You...


## Hello World: Your First Agent Execution

Now let's put it all together and create a simple "Hello World" research agent that actually performs a task.

In [9]:
# Create a simple research agent
hello_agent = Agent(
    role="Research Assistant",
    goal="Answer questions with clear, concise explanations",
    backstory="You are a helpful research assistant who provides accurate, "
              "well-researched answers to questions.",
    llm=llm,
    verbose=False,
    allow_delegation=False
)

# Define a task for the agent
hello_task = Task(
    description="Explain what CrewAI is and why it's useful for building AI agents.",
    expected_output="A clear, 2-3 paragraph explanation of CrewAI",
    agent=hello_agent
)

# Create a crew with this single agent
hello_crew = Crew(
    agents=[hello_agent],
    tasks=[hello_task],
    verbose=False
)

# Execute!
print("\n" + "="*80)
print("EXECUTING: Hello World Research Task")
print("="*80 + "\n")

result = hello_crew.kickoff()

print("\n" + "="*80)
print("RESULT:")
print("="*80)
print(result)

Overriding of current TracerProvider is not allowed



EXECUTING: Hello World Research Task


RESULT:
CrewAI is an open-source Python framework designed to orchestrate and coordinate multiple AI agents working together to accomplish complex tasks. It provides a structured approach to building multi-agent systems where different AI agents can collaborate, each with specialized roles, goals, and capabilities. The framework is built on top of large language models (LLMs) and enables developers to create "crews" of AI agents that can delegate tasks, share information, and work autonomously toward common objectives.

CrewAI is particularly useful for building AI agents because it simplifies the complexity of multi-agent coordination and collaboration. Instead of building everything from scratch, developers can define agents with specific roles (like "researcher," "writer," or "analyst"), assign them particular goals and backstories, and equip them with various tools and capabilities. The framework handles the communication and task delegation 

In [10]:
type(result)

crewai.crews.crew_output.CrewOutput

In [11]:
dir(result)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__',
 '__pydantic_serializer__',
 '__pydant

In [12]:
?result.token_usage

Type:           UsageMetrics
String form:    total_tokens=585 prompt_tokens=210 cached_prompt_tokens=0 completion_tokens=375 successful_requests=1
File:           ~/My Drive/DataForScience/CrewAI/.venv/lib/python3.12/site-packages/crewai/types/usage_metrics.py
Docstring:     
Model to track usage metrics for the crew's execution.

Attributes:
    total_tokens: Total number of tokens used.
    prompt_tokens: Number of tokens used in prompts.
    cached_prompt_tokens: Number of cached prompt tokens used.
    completion_tokens: Number of tokens used in completions.
    successful_requests: Number of successful requests made.
Init docstring:
Create a new model by parsing and validating input data from keyword arguments.

Raises [`ValidationError`][pydantic_core.ValidationError] if the input data cannot be
validated to form a valid model.

`self` is explicitly positional-only to allow `self` as a field name.